# The TimesFM adapter: messy data -> forecast-ready array

Zero-shot forecasters like [Google TimesFM](https://github.com/google-research/timesfm)
tokenize a clean, contiguous, finite context window. A raw series with gaps or
outliers makes tokenization fail. `tsa.adapters.to_timesfm()` audits, repairs, and
returns a plain `float32` numpy array — and **verifies it is finite** before
returning, so a NaN never reaches the model.

It adds **no `timesfm` dependency**: the adapter only produces numpy; the model
call stays in your code. Here we use the real OGDC `Price` series.

In [1]:
import numpy as np
import pandas as pd
import tsauditor as tsa

raw = pd.read_csv("../ogdc_leakage_case/ogdc_with_regimes.csv", index_col="Date", parse_dates=True)
price = raw[["Price"]].dropna()

# A realistic mess: a data-feed gap and a fat outlier.
dirty = price.copy()
dirty.iloc[50:60] = np.nan          # 10-row collection gap
dirty.iloc[200] = dirty.iloc[200] * 5  # fat outlier
print("rows:", len(dirty), "| NaNs:", int(dirty["Price"].isna().sum()))

rows: 1537 | NaNs: 10


## One call: audit + repair + format

`return_report=True` also hands back the `GuardReport`, so the repair is never a
black box — you can see exactly what was cleaned.

In [2]:
array, report = tsa.adapters.to_timesfm(
    dirty, target_col="Price", domain="finance", return_report=True
)

print("dtype        :", array.dtype)
print("input rows   :", len(dirty))
print("array length :", len(array), "(truncated to the most recent context_len)")
print("all finite   :", bool(np.isfinite(array).all()))
print("repaired      :", sorted({e["action"] for e in report.last_fixes}))

dtype        : float32
input rows   : 1537
array length : 1024 (truncated to the most recent context_len)
all finite   : True
repaired      : ['clip_outliers', 'clip_spikes', 'impute_interpolate']


The 1537-row series comes back as a **1024-point** array — the adapter keeps the
most recent `context_len` points (default 1024; raise it for more history, TimesFM
2.5 handles up to 16k). The gap and the outlier are gone, and the array is finite.

## The safety net: it refuses to emit a NaN

Repair only touches *flagged* problems. A lone, unclustered NaN isn't flagged, so
it would survive — and silently crash tokenization. The adapter checks finiteness
and raises instead, telling you to look before you forecast.

In [3]:
stray = price.copy()
stray.iloc[500] = np.nan   # a single, unflagged NaN

try:
    tsa.adapters.to_timesfm(stray, target_col="Price")
except ValueError as e:
    print("raised as expected:")
    print(" ", str(e)[:130], "...")

raised as expected:
  'Price' still has 1 non-finite value(s) after repair, so it cannot be fed to TimesFM. Inspect the missing values (e.g. lone/unflag ...


## Handing the array to TimesFM

With `pip install timesfm[torch]`, the model call looks like this (left un-run
here so the notebook needs only tsauditor):

```python
import timesfm

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)
model.compile(timesfm.ForecastConfig(max_context=1024, max_horizon=64))

point_forecast, quantiles = model.forecast(horizon=24, inputs=[array])
```

The separation is the point: tsauditor owns the messy-data step and guarantees a
finite array; TimesFM owns the forecast.